In [ ]:
!pip install numpy matplotlib rasterio scikit-learn albumentations torch torchvision

In [ ]:
import os
import time
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
import rasterio
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [ ]:
def get_transforms(phase='train', num_channels=4):
    mean = [0.0] * num_channels
    std  = [1.0] * num_channels # Here no normalization actually happen
    if phase == 'train':
        return A.Compose([
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.Normalize(mean=mean, std=std),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Normalize(mean=mean, std=std),
            ToTensorV2()
        ])

class CloudMaskDataset(Dataset):
    def __init__(self, img_ids, img_dir, mask_dir, bands=[1,2,3,4], transforms=None):
        self.ids = img_ids
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.bands = bands
        self.transforms = transforms

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_path = os.path.join(self.img_dir, f"{img_id}.tif")
        mask_path = os.path.join(self.mask_dir, f"{img_id}.tif")

        with rasterio.open(img_path) as src:
            img = src.read(self.bands).astype(np.float32)
        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.float32)
        mask = (mask > 0).astype(np.float32)

        img = np.transpose(img, (1,2,0))
        mask = mask[..., None]
        if self.transforms:
            aug = self.transforms(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']

        # Ensuring that the output mask is 1xHxW
        if mask.ndim == 3 and mask.shape[-1] == 1:
            mask = mask.permute(2,0,1)
        elif mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return img, mask


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64); self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128); self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256); self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(256, 512); self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2); self.conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2); self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2); self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2); self.conv1 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_channels, 1)
    def forward(self, x):
        d1, p1 = self.down1(x), self.pool1(self.down1(x))
        d2, p2 = self.down2(p1), self.pool2(self.down2(p1))
        d3, p3 = self.down3(p2), self.pool3(self.down3(p2))
        d4, p4 = self.down4(p3), self.pool4(self.down4(p3))
        bn = self.bottleneck(p4)
        u4 = self.up4(bn); c4 = self.conv4(torch.cat([u4, d4], dim=1))
        u3 = self.up3(c4); c3 = self.conv3(torch.cat([u3, d3], dim=1))
        u2 = self.up2(c3); c2 = self.conv2(torch.cat([u2, d2], dim=1))
        u1 = self.up1(c2); c1 = self.conv1(torch.cat([u1, d1], dim=1))
        return torch.sigmoid(self.outc(c1))

def dice_coef(preds, targets, smooth=1):
    p, t = preds.view(-1), targets.view(-1)
    inter = (p * t).sum()
    return (2*inter + smooth) / (p.sum() + t.sum() + smooth)

def train_epoch(model, loader, optimizer, device, criterion):
    model.train(); total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval(); total_dice=0; mis_preds=[]
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        bp = (preds>0.5).float()
        total_dice += dice_coef(bp, masks)
        for im, gt, pr in zip(imgs, masks, bp):
            if not torch.equal(pr, gt): mis_preds.append((im.cpu(), gt.cpu(), pr.cpu()))
    n = len(loader)
    return total_dice/n, mis_preds

In [ ]:
if __name__=='__main__':
    DATA_DIR = '/kaggle/input/cloud-masking-dataset/content/train/data'
    MASK_DIR = '/kaggle/input/cloud-masking-dataset/content/train/masks'
    img_paths = glob(os.path.join(DATA_DIR, '*.tif')) + glob(os.path.join(DATA_DIR, '*.tiff'))
    ids = [os.path.splitext(os.path.basename(p))[0] for p in img_paths]
    train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=42)

    train_ds = CloudMaskDataset(train_ids, DATA_DIR, MASK_DIR, transforms=get_transforms('train',4))
    val_ds   = CloudMaskDataset(val_ids,   DATA_DIR, MASK_DIR, transforms=get_transforms('val',4))
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = UNet(in_channels=4, out_channels=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss()

    best_dice = 0
    EPOCHS = 15
    for epoch in range(1, EPOCHS+1):
        start = time.time()
        print(f"Entering epoch number: {epoch} at time: {start}")
        tr_loss = train_epoch(model, train_loader, optimizer, device, criterion)
        val_dice, mis_preds = eval_model(model, val_loader, device)
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Dice: {val_dice:.4f}")
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), '/kaggle/working/best_unet.pkl')
            print("New best model saved.")

    torch.save(model.state_dict(), '/kaggle/working/final_unet.pkl')
    print("Final model has been saved")

    for img, gt, pr in mis_preds[:10]:
        fig, ax = plt.subplots(1,3,figsize=(12,4))
        ax[0].imshow(np.transpose(img.numpy(), (1,2,0))); ax[0].set_title('Image')
        ax[1].imshow(gt.numpy()[0], cmap='gray'); ax[1].set_title('GT')
        ax[2].imshow(pr.numpy()[0], cmap='gray'); ax[2].set_title('Pred')
        plt.show()